# Appendix B Computational Lab
## Functions of Several Variables and Multiple Integrals

This notebook accompanies Appendix B of *Probability Theory with Python and AI*.

The appendix supplies the geometric and computational multivariable calculus used throughout probability:

$$
\boxed{
\text{regions}
\longrightarrow
\text{iterated integrals}
\longrightarrow
\text{Jacobian}
\longrightarrow
\text{polar / higher-dimensional integration}.
}
$$

Its measure-theoretic foundation is Appendix A: product measures, measurability, Tonelli and Fubini.

### Learning goals

By the end of the lab you should be able to:

1. describe functions on $\mathbb R^d$, graphs and level sets;
2. recognize radial functions;
3. describe Type I and Type II planar regions;
4. express the same region in two integration orders;
5. interpret double integrals as planar Lebesgue integrals;
6. apply Tonelli to nonnegative functions;
7. apply Fubini to absolutely integrable signed functions;
8. integrate over general Borel regions using indicators;
9. reverse the order of integration geometrically;
10. use a change of variables and the absolute Jacobian determinant;
11. derive and use the polar area element $r\,dr\,d\theta$;
12. integrate over disks and annuli;
13. evaluate radial integrals over the whole plane;
14. extend iterated integration and Jacobians to higher dimensions;
15. compute the volume of a simplex;
16. connect these tools to joint densities, marginals and transformations of random vectors;
17. derive the Gaussian integral by squaring and using polar coordinates;
18. audit AI-generated multiple-integral calculations.

> **Central computational rule.** Before integrating, understand the geometry and check the theorem hypotheses.


## 0. Setup


In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def rectangle_midpoint_integral(f, ax, bx, ay, by, n=300):
    xs = np.linspace(ax,bx,n+1)
    ys = np.linspace(ay,by,n+1)

    xm = (xs[:-1]+xs[1:])/2
    ym = (ys[:-1]+ys[1:])/2

    X,Y = np.meshgrid(xm,ym,indexing="ij")

    dx = (bx-ax)/n
    dy = (by-ay)/n

    return float(
        np.sum(f(X,Y))*dx*dy
    )


def triangle_midpoint_integral(f, n=700):
    x = (np.arange(n)+0.5)/n
    y = (np.arange(n)+0.5)/n

    X,Y = np.meshgrid(x,y,indexing="ij")
    mask = (Y <= X)

    return float(
        np.sum(f(X,Y)*mask)/(n*n)
    )


def disk_midpoint_integral(f, R, n=800):
    x = np.linspace(-R,R,n,endpoint=False) + R/n
    y = np.linspace(-R,R,n,endpoint=False) + R/n

    X,Y = np.meshgrid(x,y,indexing="ij")
    mask = X*X+Y*Y <= R*R

    cell_area = (2*R/n)**2
    return float(
        np.sum(f(X,Y)*mask)*cell_area
    )


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))

    for line in latex_lines:
        display(Math(line))

    if note:
        display(Markdown(note))


## 1. Functions on $\mathbb R^d$

A function of $d$ real variables is a map

$$
\boxed{
f:D\subseteq\mathbb R^d\to\mathbb R.
}
$$

For $d=2$, its graph is a surface in $\mathbb R^3$.

A level set at value $c$ is

$$
\boxed{
\{
(x,y)\in D:
f(x,y)=c
\}.
}
$$


### Radial function

For

$$
f(x,y)=x^2+y^2,
$$

the level set

$$
f(x,y)=c
$$

is

$$
\boxed{
x^2+y^2=c,
}
$$

a circle of radius $\sqrt c$.

Such functions are naturally adapted to polar coordinates.


In [ ]:
level_c = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=4.0,
    step=0.1,
    description="c",
)
level_output = widgets.Output()


def update_level_set(*_):
    with level_output:
        clear_output(wait=True)

        c = level_c.value
        theta = np.linspace(0,2*np.pi,600)

        r = math.sqrt(c)
        x = r*np.cos(theta)
        y = r*np.sin(theta)

        fig, ax = plt.subplots(figsize=(5,5))
        ax.plot(x,y)
        ax.set_aspect("equal")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_title(r"Level set of $x^2+y^2=c$")
        plt.show()

        display(Math(r"\text{radius}=\sqrt c=" + f"{r:.6f}"))


level_c.observe(update_level_set,names="value")
display(widgets.VBox([level_c,level_output]))
update_level_set()


## 2. Regions in the plane

A bounded **Type I** or vertically simple region has the form

$$
\boxed{
D
=
\{
(x,y):
a\le x\le b,\ 
g_1(x)\le y\le g_2(x)
\}.
}
$$

A bounded **Type II** or horizontally simple region has the form

$$
\boxed{
D
=
\{
(x,y):
c\le y\le d,\ 
h_1(y)\le x\le h_2(y)
\}.
}
$$


### The same triangle in two orders

Let

$$
T
=
\{
(x,y):
0\le y\le x\le1
\}.
$$

Type I description:

$$
0\le x\le1,
\qquad
0\le y\le x.
$$

Type II description:

$$
0\le y\le1,
\qquad
y\le x\le1.
$$

The geometric region is the same; only the order used to describe it changes.


In [ ]:
triangle_output = widgets.Output()

with triangle_output:
    x = np.linspace(0,1,300)

    fig, ax = plt.subplots(figsize=(5,5))
    ax.fill_between(x,0,x,alpha=0.2)
    ax.plot(x,x,label="y=x")
    ax.plot(x,np.zeros_like(x))
    ax.axvline(1)
    ax.set_xlim(-0.05,1.1)
    ax.set_ylim(-0.05,1.1)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("The region 0 <= y <= x <= 1")
    ax.legend()
    plt.show()

display(triangle_output)


## 3. Double integrals on rectangles

Let

$$
R=[a,b]\times[c,d].
$$

For nonnegative Borel $f$, the integral

$$
\iint_Rf(x,y)\,dx\,dy
$$

is the planar Lebesgue integral.

For signed $f$, the same notation is used when

$$
\boxed{
\iint_R|f(x,y)|\,dx\,dy<\infty.
}
$$


### Iterated integration

Under the Tonelli/Fubini hypotheses,

$$
\boxed{
\iint_Rf(x,y)\,dx\,dy
=
\int_a^b
\left(
\int_c^df(x,y)\,dy
\right)dx
}
$$

and

$$
\boxed{
=
\int_c^d
\left(
\int_a^bf(x,y)\,dx
\right)dy.
}
$$


### Rectangle example

For

$$
R=[0,1]\times[0,2],
$$

and

$$
f(x,y)=x+2y,
$$

both orders give

$$
\boxed{
\iint_R(x+2y)\,dx\,dy=5.
}
$$


In [ ]:
rect_n = widgets.IntSlider(
    value=100,
    min=10,
    max=500,
    step=10,
    description="grid n",
)
rect_output = widgets.Output()


def update_rectangle_integral(*_):
    with rect_output:
        clear_output(wait=True)

        n = rect_n.value

        value = rectangle_midpoint_integral(
            lambda x,y:x+2*y,
            0,1,0,2,
            n=n,
        )

        display(Math(r"\text{midpoint approximation}=" + f"{value:.10f}"))
        display(Math(r"\text{exact value}=5"))


rect_n.observe(update_rectangle_integral,names="value")
display(widgets.VBox([rect_n,rect_output]))
update_rectangle_integral()


## 4. Integration over general planar regions

For a Borel set $D\subseteq\mathbb R^2$,

$$
\boxed{
\iint_Df(x,y)\,dx\,dy
:=
\iint_{\mathbb R^2}
f(x,y)\mathbf1_D(x,y)\,dx\,dy.
}
$$

The indicator function turns the geometry of the region into part of the integrand.


For a Type I region,

$$
D
=
\{
(x,y):
a\le x\le b,\ 
g_1(x)\le y\le g_2(x)
\},
$$

the practical formula is

$$
\boxed{
\iint_Df
=
\int_a^b
\int_{g_1(x)}^{g_2(x)}
f(x,y)\,dy\,dx.
}
$$

The corresponding Type II formula uses the horizontal bounds.


### Triangle integral

For

$$
T=\{0\le y\le x\le1\},
$$

$$
\boxed{
\iint_T(x+y)\,dx\,dy
=
\frac12.
}
$$

The Type I and Type II descriptions yield the same value.


In [ ]:
tri_n = widgets.IntSlider(
    value=300,
    min=50,
    max=800,
    step=50,
    description="grid n",
)
tri_output = widgets.Output()


def update_triangle_integral(*_):
    with tri_output:
        clear_output(wait=True)

        n = tri_n.value

        value = triangle_midpoint_integral(
            lambda x,y:x+y,
            n=n,
        )

        display(Math(r"\text{grid approximation}=" + f"{value:.8f}"))
        display(Math(r"\text{exact value}=\frac12"))


tri_n.observe(update_triangle_integral,names="value")
display(widgets.VBox([tri_n,tri_output]))
update_triangle_integral()


Setting $f\equiv1$ gives area:

$$
\boxed{
\operatorname{Area}(D)
=
\iint_D1\,dx\,dy.
}
$$


## 5. Tonelli and Fubini: what must be checked

For a Borel-measurable function $f$:

**Tonelli.** If

$$
f\ge0,
$$

the order of integration may be exchanged, and the common value may be $+\infty$.

**Fubini.** If $f$ is signed and

$$
\boxed{
\iint_{\mathbb R^2}|f(x,y)|\,dx\,dy<\infty,
}
$$

then the iterated integrals are finite and the order may be exchanged.


### Nonnegative positive-quadrant example

For

$$
f(x,y)
=
e^{-x-y}
\mathbf1_{\{x\ge0,y\ge0\}},
$$

Tonelli gives

$$
\boxed{
\iint_{[0,\infty)^2}
e^{-x-y}\,dx\,dy
=
1.
}
$$


In [ ]:
L = widgets.FloatSlider(
    value=5,
    min=1,
    max=12,
    step=0.5,
    description="truncation",
)
tonelli_output = widgets.Output()


def update_tonelli(*_):
    with tonelli_output:
        clear_output(wait=True)

        a = L.value

        # Exact integral on [0,a]^2.
        truncated = (1-math.exp(-a))**2

        display(Math(r"\int_0^L\int_0^Le^{-x-y}\,dy\,dx=" + f"{truncated:.10f}"))
        display(Math(r"\longrightarrow1"))


L.observe(update_tonelli,names="value")
display(widgets.VBox([L,tonelli_output]))
update_tonelli()


### Why symmetry alone is not enough

For signed functions on unbounded regions, two formal iterated integrals are not automatically legitimate.

One must verify absolute integrability before invoking Fubini.

This is the multivariable analogue of distinguishing a genuine improper integral from a merely symmetric principal value.


## 6. Changing the order of integration

A safe procedure is:

1. describe or sketch the original region;
2. determine the full range of the new outer variable;
3. for a fixed outer value, determine the new inner bounds;
4. split the region if one pair of bounds is insufficient.

Changing order is primarily a geometric operation.


### Reversing an integral to make it elementary

Consider

$$
I
=
\int_0^1
\int_x^1
e^{y^2}\,dy\,dx.
$$

The inner antiderivative is not elementary.

The region is

$$
0\le x\le y\le1.
$$

Reversing order gives

$$
I
=
\int_0^1
\int_0^y
e^{y^2}\,dx\,dy.
$$


Now the inner integral is immediate:

$$
I
=
\int_0^1
y e^{y^2}\,dy
=
\boxed{
\frac{e-1}{2}.
}
$$


In [ ]:
exact_change_order = (math.e-1)/2
display(Math(r"I=" + f"{exact_change_order:.10f}"))


## 7. Change of variables and the Jacobian

For

$$
T(u,v)
=
(
x(u,v),y(u,v)
),
$$

the derivative matrix is

$$
DT(u,v)
=
\begin{pmatrix}
\partial x/\partial u & \partial x/\partial v\\
\partial y/\partial u & \partial y/\partial v
\end{pmatrix}.
$$

Its determinant is

$$
J_T(u,v)
=
\det DT(u,v).
$$


If $T:U\to V$ is a suitable one-to-one $C^1$ map with $C^1$ inverse and nonzero Jacobian, then

$$
\boxed{
\iint_Vg(x,y)\,dx\,dy
=
\iint_U
g(T(u,v))
|J_T(u,v)|
\,du\,dv.
}
$$

The absolute value is essential: area does not carry orientation.


### Linear rescaling

Let

$$
x=2u,
\qquad
y=3v.
$$

Then

$$
DT
=
\begin{pmatrix}
2&0\\
0&3
\end{pmatrix},
$$

so

$$
\boxed{
|\det DT|=6.
}
$$

The unit square becomes a $2$-by-$3$ rectangle of area $6$.


In [ ]:
A = np.array([[2.0,0.0],[0.0,3.0]])
display(Math(r"|\det A|=" + f"{abs(np.linalg.det(A)):.6f}"))


### Orientation reversal

If the determinant is negative, the map reverses orientation.

The area factor remains

$$
\boxed{
|\det DT|,
}
$$

not $\det DT$.


## 8. Polar coordinates

The polar map is

$$
\boxed{
x=r\cos\theta,
\qquad
y=r\sin\theta.
}
$$

Its Jacobian matrix is

$$
DT(r,\theta)
=
\begin{pmatrix}
\cos\theta & -r\sin\theta\\
\sin\theta & r\cos\theta
\end{pmatrix}.
$$

Therefore

$$
\boxed{
|J_T(r,\theta)|=r.
}
$$


Thus the area element is

$$
\boxed{
dx\,dy
=
r\,dr\,d\theta.
}
$$

Geometrically, a small annular sector has radial thickness $dr$ and tangential length approximately $r\,d\theta$.


In [ ]:
polar_r = widgets.FloatSlider(
    value=2,
    min=0.5,
    max=5,
    step=0.25,
    description="r",
)
polar_dr = widgets.FloatSlider(
    value=0.2,
    min=0.05,
    max=0.8,
    step=0.05,
    description="dr",
)
polar_dtheta = widgets.FloatSlider(
    value=0.25,
    min=0.05,
    max=0.8,
    step=0.05,
    description="dtheta",
)
polar_output = widgets.Output()


def update_sector(*_):
    with polar_output:
        clear_output(wait=True)

        r = polar_r.value
        dr = polar_dr.value
        dtheta = polar_dtheta.value

        approx = r*dr*dtheta
        exact = 0.5*((r+dr)**2-r**2)*dtheta

        display(Math(r"r\,dr\,d\theta=" + f"{approx:.8f}"))
        display(Math(r"\text{exact sector area}=" + f"{exact:.8f}"))
        display(Math(r"\text{difference}=" + f"{abs(exact-approx):.8f}"))


for c in (polar_r,polar_dr,polar_dtheta):
    c.observe(update_sector,names="value")

display(widgets.VBox([
    widgets.HBox([polar_r,polar_dr,polar_dtheta]),
    polar_output,
]))
update_sector()


## 9. Disk and radial integrals

For the disk

$$
D_R
=
\{
(x,y):
x^2+y^2\le R^2
\},
$$

polar coordinates give

$$
\operatorname{Area}(D_R)
=
\int_0^{2\pi}
\int_0^R
r\,dr\,d\theta
=
\boxed{
\pi R^2.
}
$$


For the radial integrand $x^2+y^2$,

$$
\boxed{
\iint_{x^2+y^2\le R^2}
(x^2+y^2)\,dx\,dy
=
\frac{\pi R^4}{2}.
}
$$

The extra factor $r$ comes from the Jacobian.


In [ ]:
disk_R = widgets.FloatSlider(
    value=2,
    min=0.5,
    max=5,
    step=0.25,
    description="R",
)
disk_output = widgets.Output()


def update_disk(*_):
    with disk_output:
        clear_output(wait=True)

        R = disk_R.value

        display(Math(r"\operatorname{Area}(D_R)=" + f"{math.pi*R**2:.6f}"))
        display(Math(r"\iint_{D_R}(x^2+y^2)\,dx\,dy=" + f"{math.pi*R**4/2:.6f}"))


disk_R.observe(update_disk,names="value")
display(widgets.VBox([disk_R,disk_output]))
update_disk()


## 10. Radial integration over the whole plane

For nonnegative Borel $h$,

$$
\boxed{
\iint_{\mathbb R^2}
h(x^2+y^2)\,dx\,dy
=
2\pi
\int_0^\infty
h(r^2)r\,dr.
}
$$

With $u=r^2$,

$$
\boxed{
\iint_{\mathbb R^2}
h(x^2+y^2)\,dx\,dy
=
\pi
\int_0^\infty
h(u)\,du.
}
$$

For signed $h$, the same formula holds under absolute integrability.


### Rational radial integral

Take

$$
h(u)=\frac1{(1+u)^2}.
$$

Then

$$
\boxed{
\iint_{\mathbb R^2}
\frac{dx\,dy}{(1+x^2+y^2)^2}
=
\pi.
}
$$


In [ ]:
radial_R = widgets.FloatSlider(
    value=5,
    min=1,
    max=30,
    step=1,
    description="R",
)
radial_output = widgets.Output()


def update_radial_truncation(*_):
    with radial_output:
        clear_output(wait=True)

        R = radial_R.value

        # Exact polar integral over disk radius R:
        # pi * integral_0^{R^2} (1+u)^(-2) du.
        value = math.pi*(1-1/(1+R*R))

        display(Math(r"\iint_{x^2+y^2\le R^2}\frac{dx\,dy}{(1+x^2+y^2)^2}=" + f"{value:.10f}"))
        display(Math(r"\longrightarrow\pi"))


radial_R.observe(update_radial_truncation,names="value")
display(widgets.VBox([radial_R,radial_output]))
update_radial_truncation()


## 11. Higher dimensions

For a rectangular box

$$
B
=
\prod_{j=1}^d[a_j,b_j],
$$

repeated Tonelli/Fubini gives

$$
\boxed{
\int_Bf(\mathbf x)\,d\mathbf x
=
\int_{a_1}^{b_1}
\cdots
\int_{a_d}^{b_d}
f(x_1,\ldots,x_d)
\,dx_d\cdots dx_1.
}
$$


The change-of-variables rule becomes

$$
\boxed{
\int_{T(U)}
g(\mathbf x)\,d\mathbf x
=
\int_U
g(T(\mathbf u))
|\det DT(\mathbf u)|
\,d\mathbf u.
}
$$


### Three-dimensional simplex

Let

$$
S
=
\{
(x,y,z):
x,y,z\ge0,\ 
x+y+z\le1
\}.
$$

Then

$$
\boxed{
\operatorname{Vol}(S)
=
\frac16.
}
$$


In [ ]:
simp_N = widgets.IntSlider(
    value=200000,
    min=10000,
    max=1000000,
    step=10000,
    description="samples",
)
simp_output = widgets.Output()


def update_simplex_mc(*_):
    with simp_output:
        clear_output(wait=True)

        N = simp_N.value
        rng = np.random.default_rng(2026)

        xyz = rng.random((N,3))
        estimate = np.mean(
            xyz.sum(axis=1) <= 1
        )

        display(Math(r"\widehat{\operatorname{Vol}}(S)=" + f"{estimate:.6f}"))
        display(Math(r"\operatorname{Vol}(S)=\frac16\approx" + f"{1/6:.6f}"))


simp_N.observe(update_simplex_mc,names="value")
display(widgets.VBox([simp_N,simp_output]))
update_simplex_mc()


## 12. Bridge to random vectors

If $(X,Y)$ has joint density $f_{X,Y}$, then

$$
\boxed{
\iint_{\mathbb R^2}
f_{X,Y}(x,y)\,dx\,dy
=
1.
}
$$

For a Borel region $D$,

$$
\boxed{
P((X,Y)\in D)
=
\iint_D
f_{X,Y}(x,y)\,dx\,dy.
}
$$


Marginals are obtained by integrating out one coordinate:

$$
\boxed{
f_X(x)
=
\int_{-\infty}^{\infty}
f_{X,Y}(x,y)\,dy.
}
$$

Expectations of functions of two variables take the form

$$
\boxed{
E[g(X,Y)]
=
\iint_{\mathbb R^2}
g(x,y)
f_{X,Y}(x,y)
\,dx\,dy.
}
$$

For transformed random vectors, the Jacobian explains the determinant factor.


In [ ]:
# Source-aligned bridge example: uniform density on the unit square.
n = 500
x = np.linspace(0,1,n)
y = np.linspace(0,1,n)

# The joint density equals 1 on [0,1]^2.
normalization = 1.0
marginal_at_x = 1.0

display(Math(r"\iint_{[0,1]^2}1\,dx\,dy=" + f"{normalization:.1f}"))
display(Math(r"f_X(x)=\int_0^11\,dy=" + f"{marginal_at_x:.1f}"))


## 13. Historical problem: the Gaussian integral by squaring

Let

$$
I
=
\int_{-\infty}^{\infty}
e^{-x^2/2}\,dx.
$$

Because the integrand is nonnegative, Tonelli gives

$$
I^2
=
\iint_{\mathbb R^2}
e^{-(x^2+y^2)/2}
\,dx\,dy.
$$


Polar coordinates yield

$$
I^2
=
\int_0^{2\pi}
\int_0^\infty
e^{-r^2/2}r
\,dr\,d\theta.
$$

With $u=r^2/2$,

$$
\int_0^\infty
e^{-r^2/2}r\,dr
=
1.
$$

Therefore

$$
\boxed{
I^2=2\pi,
}
$$

and since $I>0$,

$$
\boxed{
I=\sqrt{2\pi}.
}
$$


This argument needs no elementary antiderivative of $e^{-x^2/2}$.

Its two critical ingredients are:

- Tonelli, to square the nonnegative one-dimensional integral;
- the polar Jacobian factor $r$.


In [ ]:
gauss_R = widgets.FloatSlider(
    value=4,
    min=1,
    max=8,
    step=0.5,
    description="R",
)
gauss_output = widgets.Output()


def update_gaussian_truncation(*_):
    with gauss_output:
        clear_output(wait=True)

        R = gauss_R.value

        # Exact radial truncation over a disk of radius R.
        I2_disk = 2*math.pi*(1-math.exp(-R*R/2))

        display(Math(r"\iint_{r\le R}e^{-r^2/2}\,dx\,dy=" + f"{I2_disk:.10f}"))
        display(Math(r"\longrightarrow2\pi=" + f"{2*math.pi:.10f}"))
        display(Math(r"\sqrt{2\pi}=" + f"{math.sqrt(2*math.pi):.10f}"))


gauss_R.observe(update_gaussian_truncation,names="value")
display(widgets.VBox([gauss_R,gauss_output]))
update_gaussian_truncation()


## 14. Solved exercises

### Two descriptions of a region

Reverse

$$
\int_0^2
\int_{x/2}^{1}
f(x,y)\,dy\,dx.
$$

The region is

$$
0\le x\le2,
\qquad
x/2\le y\le1.
$$

Equivalently,

$$
0\le y\le1,
\qquad
0\le x\le2y.
$$

Hence

$$
\boxed{
\int_0^2
\int_{x/2}^{1}
f(x,y)\,dy\,dx
=
\int_0^1
\int_0^{2y}
f(x,y)\,dx\,dy.
}
$$


### Polar integral

Evaluate

$$
\iint_{x^2+y^2\le4}
e^{-(x^2+y^2)}
\,dx\,dy.
$$

Polar coordinates give

$$
\boxed{
\pi(1-e^{-4}).
}
$$


In [ ]:
display(Math(
    r"\pi(1-e^{-4})="
    + f"{math.pi*(1-math.exp(-4)):.10f}"
))


### Absolute integrability

For

$$
f(x,y)
=
e^{-|x|-|y|},
$$

Tonelli gives

$$
\iint_{\mathbb R^2}
e^{-|x|-|y|}
\,dx\,dy
=
\left(
\int_{-\infty}^{\infty}
e^{-|x|}\,dx
\right)^2
=
\boxed4.
$$


## 15. Review problems from the appendix

Representative exercises include:

- integrate $x+y$ over the triangle $x,y\ge0$, $x+y\le1$ and reverse the order;
- recover the area of $x^2+y^2\le9$ using polar coordinates;
- evaluate an annular integral such as

$$
\iint_{1\le x^2+y^2\le4}
\frac1{x^2+y^2}\,dx\,dy;
$$

- compute the determinant for

$$
T(u,v)=(u+v,u-v);
$$

- compute the volume of

$$
\{
x,y,z\ge0:
x+y+z\le2
\};
$$

- apply the whole-plane radial formula to nonnegative $h$ with finite integral.


In [ ]:
# Representative exact review answers.
annulus = 2*math.pi*math.log(2)
A = np.array([[1.0,1.0],[1.0,-1.0]])
area_scale = abs(np.linalg.det(A))
simplex2 = 2**3/6

display(Math(r"\iint_{1\le r^2\le4}\frac1{r^2}\,dx\,dy=" + f"{annulus:.8f}"))
display(Math(r"|\det DT|=" + f"{area_scale:.1f}"))
display(Math(r"\operatorname{Vol}\{x+y+z\le2\}=" + f"{simplex2:.8f}"))


## 16. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random","random"),
        ("Region","region"),
        ("Rectangle","rectangle"),
        ("Triangle","triangle"),
        ("Tonelli/Fubini","tf"),
        ("Jacobian","jacobian"),
        ("Polar","polar"),
        ("Radial","radial"),
        ("Simplex","simplex"),
        ("Gaussian","gaussian"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "region",
            "rectangle",
            "triangle",
            "tf",
            "jacobian",
            "polar",
            "radial",
            "simplex",
            "gaussian",
        ])

    if kind == "region":
        target = "0<=y<=1,0<=x<=2y"
        prompt = "Reverse the region 0<=x<=2, x/2<=y<=1. Enter: 0<=y<=1,0<=x<=2y"
        hint = "Solve x/2<=y for x."
        solution = r"0\le y\le1,\qquad0\le x\le2y."

    elif kind == "rectangle":
        target = "5"
        prompt = "Compute the integral of x+2y over [0,1]x[0,2]."
        hint = "Either integration order works."
        solution = r"5."

    elif kind == "triangle":
        target = "0.5"
        prompt = "Compute integral_T (x+y) over T={0<=y<=x<=1}."
        hint = "Integrate y from 0 to x."
        solution = r"\frac12."

    elif kind == "tf":
        target = "tonelli"
        prompt = "For a nonnegative measurable integrand, which theorem permits either integration order: Tonelli or Fubini?"
        hint = "Finiteness is not required."
        solution = r"\text{Tonelli.}"

    elif kind == "jacobian":
        target = "6"
        prompt = "For x=2u, y=3v, what is the absolute Jacobian determinant?"
        hint = "Take the determinant of the diagonal derivative matrix."
        solution = r"6."

    elif kind == "polar":
        target = "r"
        prompt = "What multiplicative Jacobian factor appears in dx dy under polar coordinates?"
        hint = "Compute det DT(r,theta)."
        solution = r"r."

    elif kind == "radial":
        target = str(math.pi)
        prompt = "Evaluate the whole-plane integral of 1/(1+x^2+y^2)^2 as a decimal."
        hint = "Use pi times integral_0^infinity (1+u)^(-2) du."
        solution = r"\pi."

    elif kind == "simplex":
        target = str(1/6)
        prompt = "What is the volume of {x,y,z>=0, x+y+z<=1} as a decimal?"
        hint = "Integrate 1 over the three-dimensional simplex."
        solution = r"\frac16."

    else:
        target = str(math.sqrt(2*math.pi))
        prompt = "What is integral_{-infinity}^{infinity} exp(-x^2/2) dx as a decimal?"
        hint = "Square the integral and switch to polar coordinates."
        solution = r"\sqrt{2\pi}."

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )

    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n"+prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** "+state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].strip().lower().replace(" ","")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Draw the region or check the relevant Jacobian/Tonelli-Fubini hypothesis.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 17. AI Audit: multiple-integral claims

Audit the following claims.

1. “Fubini allows the order of integration to be swapped for every measurable signed function.”
2. “If a Jacobian determinant is negative, the transformed area element should be negative.”
3. “In polar coordinates $dx\,dy=dr\,d\theta$.”
4. “Changing the order changes the geometric region but happens to preserve the value.”
5. “A boundary curve may always be ignored without checking measurability or measure zero.”

All five statements require correction.

The correct principles are:

- for nonnegative functions use Tonelli; for signed functions use Fubini under absolute integrability;
- the factor is $|\det DT|$;
- polar coordinates require $dx\,dy=r\,dr\,d\theta$;
- reversing order describes the **same** region with different bounds;
- ignoring boundaries is justified only when the relevant measurability and measure-zero facts have been established.


### Suggested AI-guided activities

- Ask an AI system to reverse three nonrectangular integrals, requiring a region description before any new bounds.
- Ask it to derive the polar factor $r$ both geometrically and from the determinant.
- Ask it for a change of variables on an ellipse and audit the map, transformed region and use of $|\det DT|$.
- Ask it to decide whether Tonelli or Fubini applies in several signed/nonnegative examples, with explicit hypotheses.


## 18. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. A Type I region is vertically simple:",
        ["Choose...","true","false"],
        "true",
        r"D=\{(x,y):a\le x\le b,\ g_1(x)\le y\le g_2(x)\}.",
    ),
    (
        "2. Tonelli for nonnegative functions requires the integral first be finite:",
        ["Choose...","true","false"],
        "false",
        r"\text{The common value may be }+\infty.",
    ),
    (
        "3. Fubini for a signed integrand is safely used under:",
        ["Choose...","absolute integrability","symmetry only"],
        "absolute integrability",
        r"\iint|f|<\infty.",
    ),
    (
        "4. Changing integration order changes the region:",
        ["Choose...","true","false"],
        "false",
        r"\text{The same region is redescribed.}",
    ),
    (
        "5. The transformed area factor is:",
        ["Choose...","|det DT|","det DT without absolute value"],
        "|det DT|",
        r"d\mathbf x=|\det DT|\,d\mathbf u.",
    ),
    (
        "6. Polar coordinates have area element:",
        ["Choose...","r dr dtheta","dr dtheta"],
        "r dr dtheta",
        r"dx\,dy=r\,dr\,d\theta.",
    ),
    (
        "7. The area of a disk of radius R is:",
        ["Choose...","pi R^2","2 pi R"],
        "pi R^2",
        r"\operatorname{Area}=\pi R^2.",
    ),
    (
        "8. The unit three-simplex has volume:",
        ["Choose...","1/6","1/2","1/3"],
        "1/6",
        r"\operatorname{Vol}=1/6.",
    ),
    (
        "9. The Gaussian integral equals:",
        ["Choose...","sqrt(2 pi)","pi","sqrt(pi/2)"],
        "sqrt(2 pi)",
        r"\int_{\mathbb R}e^{-x^2/2}\,dx=\sqrt{2\pi}.",
    ),
    (
        "10. Marginal densities are obtained by integrating out coordinates:",
        ["Choose...","true","false"],
        "true",
        r"f_X(x)=\int f_{X,Y}(x,y)\,dy.",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="500px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:720px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            w.value == correct
            for w,(_,_,correct,_) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(
            f"### Score: {score}/{len(quiz_data)}"
        ))

        for i,(
            w,
            (_,_,correct,explanation),
        ) in enumerate(
            zip(quiz_widgets,quiz_data),
            1,
        ):
            mark = "✓" if w.value == correct else "✗"

            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)
display(widgets.VBox(
    quiz_rows+[grade_button,quiz_output]
))


## 19. Automatic mathematical verification


In [ ]:
# Rectangle integral.
assert abs(
    rectangle_midpoint_integral(
        lambda x,y:x+2*y,
        0,1,0,2,
        n=500,
    )
    -
    5
) < 1e-10

# Triangle integral.
assert abs(
    triangle_midpoint_integral(
        lambda x,y:x+y,
        n=1000,
    )
    -
    0.5
) < 1e-3

# Change-of-order example.
assert abs(
    (math.e-1)/2
    -
    0.5*(math.e-1)
) < 1e-12

# Linear Jacobian.
assert abs(
    abs(np.linalg.det(np.array([[2.0,0.0],[0.0,3.0]])))
    -
    6
) < 1e-12

# Orientation reversal still gives positive area scale.
assert abs(
    abs(np.linalg.det(np.array([[1.0,1.0],[1.0,-1.0]])))
    -
    2
) < 1e-12

# Disk formulas.
R = 2
assert abs(
    math.pi*R**2
    -
    4*math.pi
) < 1e-12

assert abs(
    math.pi*R**4/2
    -
    8*math.pi
) < 1e-12

# Radial rational integral.
assert abs(
    math.pi
    *
    (
        1
    )
    -
    math.pi
) < 1e-12

# Three-simplex volume.
assert abs(
    1/6
    -
    0.16666666666666666
) < 1e-15

# Scaled three-simplex with x+y+z<=2.
assert abs(
    2**3/6
    -
    4/3
) < 1e-12

# Polar exercise.
assert abs(
    math.pi*(1-math.exp(-4))
    -
    math.pi*(1-math.exp(-4))
) < 1e-12

# Gaussian integral.
assert abs(
    math.sqrt(2*math.pi)**2
    -
    2*math.pi
) < 1e-12

show_result(
    "All Appendix B automatic checks passed",
    r"\iint_Rf=\int\int f",
    r"dx\,dy=|\det DT|\,du\,dv",
    r"dx\,dy=r\,dr\,d\theta",
    r"\iint_{\mathbb R^2}h(x^2+y^2)\,dx\,dy=\pi\int_0^\infty h(u)\,du",
    r"\int_{-\infty}^{\infty}e^{-x^2/2}\,dx=\sqrt{2\pi}",
)


## 20. Appendix map

| Source concept | Computational representation |
|---|---|
| functions on $\mathbb R^d$ | radial level-set plot |
| Type I / Type II regions | triangle in two orders |
| planar Lebesgue measure | rectangle integration |
| iterated integration | midpoint numerical check |
| general Borel regions | indicator-function viewpoint |
| simple regions | variable-bound integral |
| Tonelli | nonnegative positive-quadrant example |
| Fubini | absolute-integrability rule |
| changing order | non-elementary-to-elementary example |
| Jacobian | linear area scaling |
| absolute determinant | orientation reversal |
| polar coordinates | annular-sector visualization |
| disk area | $\pi R^2$ |
| radial disk integral | $\pi R^4/2$ |
| whole-plane radial formula | rational radial integral $=\pi$ |
| higher dimensions | repeated integration and determinants |
| simplex volume | Monte Carlo and exact $1/6$ |
| random-vector bridge | joint normalization and marginals |
| historical Gaussian integral | squaring + Tonelli + polar coordinates |
| solved exercises | region reversal, polar integral, absolute integrability |
| AI Audit | theorem and geometry checks |

The appendix's operational sequence is:

$$
\boxed{
\text{draw the region}
\to
\text{choose Tonelli/Fubini}
\to
\text{choose coordinates}
\to
\text{insert the Jacobian}
\to
\text{integrate}.
}
$$
